# 🚀 MDAAR v2.0: Multi-role Dynamic Architecture for Automated Repositories
### المنظومة الهندسية الشاملة لتوثيق المشاريع واسترجاع المعرفة بالذكاء الاصطناعي
---
**أبرز المميزات:**
1. **قراءة 100% من ملفات المشروع بلا حدود للسياق:** فحص شامل لكل ملف وقطعة برمجية (`Map-Reduce`) مع تعقيم الأسرار (`SecretSanitizer`).
2. **التبديل التلقائي بين موديلات Gemini المجانية (`auto-free-chain`):** يبدأ بـ `gemini-3.8-flash` وينتقل تلقائياً عبر `gemini-3.6-flash` ➔ `gemini-3.5-flash-lite` ➔ `gemini-3.1-flash-lite` ➔ `gemini-3-flash-preview` ➔ `gemma-4-31b-it` ➔ `gemma-4-26b-a4b-it`.
3. **دعم بوابات OpenAI وراوتر الكولاب الثاني (`OpenAI-Gateway`):** ربط فوري مع OpenRouter أو Cloudflare أو Portkey أو Groq أو [`free_router_colab.ipynb`](https://colab.research.google.com/github/braadiv/MDAAR/blob/main/free_router_colab.ipynb).
4. **واجهة كولاب ذكية مشروطة (`ipywidgets`):** تُظهر فقط الحقول الخاصة باختيارك وتخفي البقية، وتوحّد المفاتيح المشتركة دون أي قوائم `AUTO` مزعجة.
5. **تثبيت تلقائي عند الحاجة:** مكتبات GPU المحلية (`transformers`/`bitsandbytes`) لا تُثبّت في البداية وتُثبّت تلقائياً فقط إذا اخترت `Local-Offline`.

## 🔑 أسرار كولاب ومتغيرات البيئة (`Colab Secrets` / `Env Vars`)
جميع الحقول تتعبأ تلقائياً من **Colab Secrets (🔑)** إن وُجدت، وتُخفي الواجهة التفاعلية أي حقول غير مرتبطة باختيارك الحالي (المفاتيح والإعدادات المشتركة مُوحّدة ولا تتكرر):

| اسم المتغير / Secret Name | الاستخدام والافتراضي |
|:---|:---|
| `MDAAR_PROJECT_NAME` | اسم المشروع (الافتراضي `MySystem`) |
| `MDAAR_GIT_SOURCE_URL` | رابط مستودع الكود المصدري Git |
| `MDAAR_SOURCE_AUTH` | مصادقة الكود (`HTTPS` أو `SSH`) |
| `MDAAR_BASE_WORKSPACE` | مجلد العمل (افتراضياً Drive إن وُجد) |
| `MDAAR_ENABLE_EXTERNAL_DOCS_REPO` | تفعيل مستودع التوثيق الخارجي (`True` / `False`) |
| `MDAAR_GIT_DOCS_URL` / `MDAAR_DOCS_AUTH` | رابط ومصادقة مستودع التوثيق الخارجي |
| `MDAAR_PLANNER_PROVIDER` | مزود مودل الخطة (`Gemini` / `OpenAI-Gateway` / `Local-Offline`) |
| `MDAAR_PLANNER_GEMINI_MODEL` / `MDAAR_EXECUTOR_GEMINI_MODEL` | مودل Gemini للخطة / التنفيذ (الافتراضي `auto-free-chain` ويبدأ بـ `gemini-3.8-flash`) |
| `MDAAR_PLANNER_OPENAI_MODELS` / `MDAAR_EXECUTOR_OPENAI_MODELS` | اسم مودل البوابة للخطة / التنفيذ (أو `OPENAI_MODEL`) |
| `MDAAR_PLANNER_LOCAL_MODEL` / `MDAAR_EXECUTOR_LOCAL_MODEL` | نموذج HuggingFace المحلي للخطة / التنفيذ |
| `MDAAR_EXECUTOR_MODE` | `same` (نفس مودل الخطة) أو `custom` (مودل مستقل) |
| `MDAAR_EXECUTOR_PROVIDER` | مزود مودل التنفيذ عند اختيار `custom` |
| `GEMINI_API_KEY` أو `MDAAR_GEMINI_KEYS` | **مشترك:** مفاتيح Gemini (يدعم عدة مفاتيح مفصولة بفاصلة) |
| `MDAAR_OPENAI_BASE_URL` أو `MDAAR_ROUTER_URL` أو `OPENAI_BASE_URL` | **مشترك:** رابط OpenRouter / Cloudflare / Portkey / راوتر الكولاب الثاني |
| `MDAAR_OPENAI_API_KEY` أو `MDAAR_ROUTER_TOKEN` أو `OPENROUTER_API_KEY` | **مشترك:** مفتاح البوابة الخارجية أو توكن راوتر الكولاب الثاني |
| `MDAAR_QUANTIZATION` / `HF_TOKEN` | **مشترك:** التكميم (`4bit`/`None`) وتوكن HuggingFace للموديلات المحلية |
| `MDAAR_DOCS_PORT` / `MDAAR_RAG_QUESTION` | منفذ MkDocs (`8000`) وسؤال المساعد الذكي الافتراضي |

## 1️⃣ تجهيز البيئة وتثبيت MDAAR وربط Google Drive
تثبيت سريع لحزمة **MDAAR** والمهارات المدمجة وأداة `cloudflared` (دون إضاعة الوقت في مكتبات GPU إلا عند اختيار نموذج محلي).

In [ ]:
#@title 1. تفعيل Drive وتثبيت محرك MDAAR السريع { display-mode: "form" }
import os, json, subprocess
from pathlib import Path

DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ تم ربط Google Drive بنجاح.')
    except Exception as e:
        print('⚠️ سيتم استخدام مسار التخزين المحلي:', e)

print('⏳ جاري تثبيت حزمة MDAAR والمهارات المدمجة...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ تثبيت cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, SkillLoader
skills_count = len(SkillLoader().list_bundled_skills())
print(f'🎉 تم تجهيز MDAAR v{__version__} مع {skills_count} مهارات مدمجة بنجاح!')

## 2️⃣ لوحة إعدادات المشروع والمستودعات (تفاعلية)
شغّل الخلية لعرض اللوحة التفاعلية: الحقول تُسحب تلقائياً من **Colab Secrets** إن وُجدت، وتظهر حقول مستودع التوثيق الخارجي فقط عند تفعيل خياره.

In [ ]:
#@title 2. لوحة إعداد المستودعات وفحص الاتصال الآمن { display-mode: "form" }
from mdaar_pipeline import launch_repo_config_ui
repo_ui = launch_repo_config_ui(lang='ar', namespace=globals(), test_on_start=True)

## 3️⃣ لوحة إعدادات النماذج والمفاتيح المشتركة (تفاعلية)
شغّل الخلية واختر المزود لكل من **الخطة** و**التنفيذ**:
- تُعرض فقط الحقول الخاصة بالمزود المختار وتختفي الحقول الأخرى تلقائياً.
- المفاتيح (`GEMINI_API_KEY`, `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `QUANTIZATION`, `HF_TOKEN`) موحّدة في قسم مشترك واحد ولا تتكرر.

In [ ]:
#@title 3. لوحة إعداد مودل الخطة ومودل التنفيذ والمفاتيح المشتركة { display-mode: "form" }
from mdaar_pipeline import launch_models_config_ui
models_ui = launch_models_config_ui(pipeline=globals().get('pipeline'), lang='ar', namespace=globals())

## 4️⃣ استنساخ الكود والتحقق الحتمي قبل البدء

In [ ]:
#@title 4. مزامنة الكود ورصد الملفات المعدلة { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 حالة الـ Commit الحالي: ({current_commit[:7] if current_commit else "Local"})')
if changed_files:
    print(f'🔍 تم رصد {len(changed_files)} ملفاً متغيراً منذ آخر توثيق:')
    for f in changed_files[:15]:
        print('  •', f)
else:
    print('ℹ️ سيتم إجراء فحص كامل ومباشر لجميع ملفات المشروع.')

## 5️⃣ الفحص الكامل 100%، الخطة، توليد الحزم وبوابة الجودة + `.rag_index/`

In [ ]:
#@title 5. تنفيذ الفحص الكامل 100% وتوليد جميع حزم التوثيق { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

def _show_map_progress(idx, total, module_name, files_desc):
    if idx == 1 or idx == total or idx % 5 == 0:
        print(f'  [{idx}/{total}] فحص الوحدة `{module_name}`: {files_desc[:90]}')

target_changes = None if FORCE_FULL_GENERATION else changed_files
run_stats = pipeline.run_multi_agent_generation(
    changed_files=target_changes,
    force_full=FORCE_FULL_GENERATION,
    progress_callback=_show_map_progress
)

if current_commit:
    pipeline.save_state(current_commit)

print('='*70)
print('📊 ملخص دورة التوثيق المكتملة:')
print(json.dumps(run_stats, indent=2, ensure_ascii=False))
print('='*70)

## 6️⃣ إطلاق بوابة التوثيق التفاعلية (MkDocs Material + Cloudflare Tunnel)

In [ ]:
#@title 6. تشغيل البوابة ومزامنة مستودع الوثائق { display-mode: "form" }
DOCS_PORT = "" #@param {type:"string"}

from mdaar_pipeline import resolve_setting

docs_port = int(resolve_setting(DOCS_PORT, ('MDAAR_DOCS_PORT',), default='8000') or 8000)
tunnel_url = pipeline.serve_and_tunnel(port=docs_port)

if tunnel_url:
    print('='*70)
    print(f'🎉 بوابة التوثيق التفاعلية جاهزة على المنفذ {docs_port} ومتاحة عبر الإنترنت:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ جاري تشغيل الخادم محلياً، يمكنك التحقق من سجلات النفق.')

sync_docs_res = pipeline.sync_docs_repo()
print('📡 مزامنة مستودع الوثائق:', sync_docs_res)

## 7️⃣ المساعد المعرفي الذكي الهجين (Hybrid BM25 RAG — `.rag_index/`)

In [ ]:
#@title 7. اسأل مساعد التوثيق الذكي حول المشروع { display-mode: "form" }
QUESTION = "" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

from mdaar_pipeline import resolve_setting

resolved_q = resolve_setting(QUESTION, ('MDAAR_RAG_QUESTION',), default='ما هي معمارية النظام والمكونات الرئيسية والجداول والواجهات الفعلية؟')

chat_engine = pipeline.get_chat_engine()
chunks_loaded = chat_engine.load_documents(prefer_saved_index=True)
print(f'📚 تم تحميل {chunks_loaded} مقطعاً معرفياً من `{chat_engine.index_dir}`')
print(f'🔍 جاري الاسترجاع والإجابة عن: "{resolved_q}" (الجمهور: {AUDIENCE})...\n')

answer = chat_engine.ask(resolved_q, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)